In [1]:
import torch
import numpy as np  
import yaml
import mlflow
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import joblib as jl
import shap
mlflow.set_tracking_uri("http://localhost:5000")

In [2]:
config = yaml.safe_load(open("/workspace/config/config.yaml"))

# -----------------------------
# Load preprocessing artifacts
# -----------------------------
experiment = mlflow.get_experiment_by_name("Default")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.stage = 'preprocessing'",
    order_by=["start_time DESC"],
    max_results=1
)

run_id = runs.iloc[0].run_id

scaler_path_X = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/scaler_X.pkl"
)
scaler_path_y = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/scaler_y.pkl"
)

train_path_X = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/train_X.pkl"
)
train_path_y = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/train_y.pkl"
)

val_path_X = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/val_X.pkl"
)
val_path_y = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/val_y.pkl"
)

# Load data
scaler_X = jl.load(scaler_path_X)
scaler_y = jl.load(scaler_path_y)

train_series_X = pd.read_pickle(train_path_X)
train_series_y = pd.read_pickle(train_path_y)

val_series_X = pd.read_pickle(val_path_X)
val_series_y = pd.read_pickle(val_path_y)

class ForecastDataset(Dataset):
    def __init__(self,series_X,series_y,look_back,horizon):
        self.series_X=series_X
        self.series_y=series_y
        self.look_back=look_back
        self.horizon=horizon
    def __len__(self):
        return len(self.series_X)-self.look_back-self.horizon+1
    def __getitem__(self,idx):
        X=self.series_X[idx:idx+self.look_back,:]
        y=self.series_y[idx+self.look_back:idx+self.look_back+self.horizon,:].squeeze(-1)
        return X,y
train_dataset=ForecastDataset(train_series_X,train_series_y,look_back=config["data"]["look_back"],horizon=config["data"]["horizon"])
train_dataloader=DataLoader(train_dataset,batch_size=config["training"]["batch_size"],shuffle=False)
val_dataset=ForecastDataset(val_series_X,val_series_y,look_back=config["data"]["look_back"],horizon=config["data"]["horizon"])
val_dataloader=DataLoader(val_dataset,batch_size=config["training"]["batch_size"],shuffle=False)

In [3]:
experiment = mlflow.get_experiment_by_name("Default")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.stage = 'training'",
    order_by=["start_time DESC"],
    max_results=1
)

run_id = runs.iloc[0].run_id
model_path = f"runs:/{run_id}/model"
model_uri = 'runs:/1ca274fa53f540b78c0796e36790d02b/model'
model = mlflow.pytorch.load_model(model_uri)

In [4]:
print(train_series_X.shape)

(1434496, 13)


In [5]:
class BaselineProvider():
    def sample(input,perspective):
        if perspective=="Global":
            return train_series_X[train_series_X[-1]==input[-1]]
        elif perspective=="Local":
            return train_series_X[(train_series_X[-1]==input[-1])&(train_series_[-4]==input[-4])&(train_series_X[-5]==input[-5])]
        else:
            return train_series_X[(train_series_X[-1]==input[-1])&(train_series_[-7]==input[-7])&(train_series_X[-6]==input[-6])]
class BaselineDataset(Dataset):
    def __init__(self,series_X,look_back):
        self.series_X=series_X
        self.look_back=look_back
    def __getitem__(self,idx):
        X=self.series_X[idx:idx+self.look_back,:]
        return X


In [10]:
import json
import torch
from captum.attr import IntegratedGradients

# --------------------------------------------------
# Setup
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.train()

ig = IntegratedGradients(model)

FEATURE_NAMES = [
    "Global_active_power",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "year",
    "quarter_sin",
    "quarter_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "weekday"
]

# --------------------------------------------------
# Calculate mean baseline from training data
# --------------------------------------------------

sum_windows = None
num_windows = 0

for train_x, _ in train_dataloader:

    train_x = train_x.float().to(device)

    if sum_windows is None:
        sum_windows = torch.zeros(
            train_x.shape[1:],      # (120, 13)
            device=device
        )

    sum_windows += train_x.sum(dim=0)
    num_windows += train_x.size(0)

mean_window = sum_windows / num_windows

# shape: (120, 13)

# --------------------------------------------------
# Result metadata
# --------------------------------------------------

metadata = {
    "num_targets": 30,
    "sequence_length": 120,
    "num_features": 13,
    "features": FEATURE_NAMES,
    "captum_method": "IntegratedGradients",
    "n_steps": 50
}

# --------------------------------------------------
# Open output file
#
# We write the JSON incrementally instead of keeping
# the entire results dictionary in RAM.
# --------------------------------------------------

with open(
    "captum.json",
    mode="w",
    encoding="utf-8"
) as file:

    # --------------------------------------------------
    # Start JSON
    # --------------------------------------------------

    file.write("{\n")

    # --------------------------------------------------
    # Write metadata
    # --------------------------------------------------

    file.write('    "metadata": ')
    json.dump(
        metadata,
        file,
        indent=4
    )

    file.write(",\n")

    # --------------------------------------------------
    # Start samples object
    # --------------------------------------------------

    file.write('    "samples": {\n')

    first_sample = True

    # --------------------------------------------------
    # Process entire validation dataset
    # --------------------------------------------------

    global_sample_id = 0

    for batch_idx, (X, y) in enumerate(val_dataloader):

        val_x = X.float().to(device)
        val_y = y.float().to(device)

        batch_size = val_x.size(0)

        # Baseline must match THIS batch size
        mean_baseline = (
            mean_window
            .unsqueeze(0)
            .expand(batch_size, -1, -1)
        )

        # ----------------------------------------------
        # Prediction
        # ----------------------------------------------

        with torch.no_grad():
            pred = model(val_x)

        # ----------------------------------------------
        # Captum
        # ----------------------------------------------

        attr = []
        delta = []

        for target_idx in range(30):

            att, delt = ig.attribute(
                val_x,
                target=target_idx,
                baselines=mean_baseline,
                return_convergence_delta=True,
                n_steps=50
            )

            attr.append(att.detach().cpu())
            delta.append(delt.detach().cpu())

        # ----------------------------------------------
        # Store each sample
        # ----------------------------------------------

        for sample_idx in range(batch_size):

            global_sample_id += 1

            sample_key = f"sample_{global_sample_id}"

            sample = {
                "sample_id": global_sample_id,
                "prediction": {},
                "real_value": {},
                "delta": {},
                "window_attribution": {}
            }

            # ------------------------------------------
            # Store all 30 prediction targets
            # ------------------------------------------

            for target_idx in range(30):

                target_key = f"target_{target_idx + 1}"

                sample["prediction"][target_key] = float(
                    pred[sample_idx, target_idx].item()
                )

                sample["real_value"][target_key] = float(
                    val_y[sample_idx, target_idx].item()
                )

                sample["delta"][target_key] = float(
                    delta[target_idx][sample_idx].item()
                )

                # --------------------------------------
                # Attribution for this target
                # --------------------------------------

                target_attr = attr[target_idx][sample_idx]

                # shape: (120, 13)

                sample["window_attribution"][target_key] = {}

                for window_idx in range(target_attr.shape[0]):

                    window_key = f"window_{window_idx + 1}"

                    window_values = target_attr[window_idx].tolist()

                    sample["window_attribution"][target_key][window_key] = {
                        feature: float(value)
                        for feature, value in zip(
                            FEATURE_NAMES,
                            window_values
                        )
                    }

            # ------------------------------------------
            # Write this sample immediately to disk
            # ------------------------------------------

            if not first_sample:
                file.write(",\n")

            file.write(
                f'        "{sample_key}": '
            )

            json.dump(
                sample,
                file,
                indent=8
            )

            first_sample = False

            # Make sure the sample is written to disk
            file.flush()

            print(
                f"Processed sample {global_sample_id}"
            )
        if global_sample_id >= 1000:
            break

    # --------------------------------------------------
    # Finish samples object
    # --------------------------------------------------

    file.write("\n    }\n")

    # --------------------------------------------------
    # Finish JSON
    # --------------------------------------------------

    file.write("}\n")
    

print(
    f"Saved {global_sample_id} validation samples "
    "to captum.json"
)

Processed sample 1
Processed sample 2
Processed sample 3
Processed sample 4
Processed sample 5
Processed sample 6
Processed sample 7
Processed sample 8
Processed sample 9
Processed sample 10
Processed sample 11
Processed sample 12
Processed sample 13
Processed sample 14
Processed sample 15
Processed sample 16
Processed sample 17
Processed sample 18
Processed sample 19
Processed sample 20
Processed sample 21
Processed sample 22
Processed sample 23
Processed sample 24
Processed sample 25
Processed sample 26
Processed sample 27
Processed sample 28
Processed sample 29
Processed sample 30
Processed sample 31
Processed sample 32
Processed sample 33
Processed sample 34
Processed sample 35
Processed sample 36
Processed sample 37
Processed sample 38
Processed sample 39
Processed sample 40
Processed sample 41
Processed sample 42
Processed sample 43
Processed sample 44
Processed sample 45
Processed sample 46
Processed sample 47
Processed sample 48
Processed sample 49
Processed sample 50
Processed

In [ ]:
# device = torch.device("cpu")

# model = model.to(device)
# val_x = val_x.cpu()
# pred = model(val_x)
# mean_baseline = mean_baseline.cpu()
# if val_x.grad is not None:
#     val_x.grad.zero_()
# val_x = val_x.detach().requires_grad_(True)
# def forward_target(x):
#     return model(x)[:, 0]
# ig = IntegratedGradients(forward_target)

# att, delta = ig.attribute(
#     val_x[0:1],
#     baselines=mean_baseline[0:1],
#     return_convergence_delta=True
# )

# print(att.sum())
# print(delta)
# baseline_pred = model(mean_baseline[0:1])

# print(baseline_pred)
# print(pred[0])
# model.zero_grad()

# output = model(val_x[0:1])[0, 0]
# output.backward()

# print(val_x.grad.min())
# print(val_x.grad.max())
# print(val_x.grad.mean())
# for name, param in model.named_parameters():
#     print(
#         name,
#         param.data.min().item(),
#         param.data.max().item(),
#         param.data.mean().item()
#     )
#     print(name, param.norm().item())

In [ ]:
# x = val_x[0:1]

# output, (h, c) = model.lstm(x)

# print(output.min())
# print(output.max())
# print(output.mean())

# print(h.min())
# print(h.max())
# print(h.mean())
# last = output[:, -1, :]

# print(last.min())
# print(last.max())
# print(last.mean())
# fc = model.fc(last)

# print(fc.min())
# print(fc.max())
# print(fc.mean())

In [ ]:
# x = val_x[0:1].clone().detach().requires_grad_(True)
# print(torch.__version__)
# y = model(x)[0,0]

# grad = torch.autograd.grad(
#     outputs=y,
#     inputs=x
# )[0]

# print(grad.min())
# print(grad.max())
# print(grad.mean())

In [ ]:
# print(torch.backends.mkldnn.enabled)
# torch.backends.mkldnn.enabled = False
# output = model(val_x[0:1])[0,0]
# output.backward()

# print(val_x.grad.min())
# print(val_x.grad.max())


In [1]:
# import torch
# import gc  # Garbage collection module

# def clear_vram():
#     """
#     Safely clears all GPU VRAM resources used by PyTorch in the current process.
#     """
#     try:
#         # Keep essential names so we don't delete them
#         keep_vars = {"torch", "gc", "clear_vram", "__name__", "__doc__", "__package__", "__loader__", "__spec__", "__annotations__", "__builtins__"}

#         # Delete all other global variables that may hold GPU tensors/models
#         for obj in list(globals().keys()):
#             if obj not in keep_vars:
#                 del globals()[obj]

#         # Force Python garbage collection
#         gc.collect()

#         # Clear PyTorch CUDA cache
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()       # Releases cached memory
#             torch.cuda.ipc_collect()       # Collects inter-process memory
#             torch.cuda.reset_peak_memory_stats()
#             torch.cuda.reset_accumulated_memory_stats()
#             print("✅ GPU VRAM cleared successfully.")
#         else:
#             print("⚠️ No CUDA-enabled GPU detected.")

#     except Exception as e:
#         print(f"Error while clearing VRAM: {e}")

# # Example usage:
# if __name__ == "__main__":
#     clear_vram()


✅ GPU VRAM cleared successfully.


In [ ]:
# import torch
# import gc
# import sys

# def analyze_vram():
#     """
#     Lists all PyTorch tensors/models in the current Python session that are using GPU VRAM.
#     Shows their type, shape, dtype, and estimated memory usage.
#     """
#     if not torch.cuda.is_available():
#         print("⚠️ No CUDA-enabled GPU detected.")
#         return

#     total_mem = 0
#     print("\n🔍 GPU VRAM Usage Report:\n")

#     # Iterate over all tracked Python objects
#     for obj in gc.get_objects():
#         try:
#             if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
#                 if obj.is_cuda:
#                     # Calculate memory in MB
#                     mem_mb = obj.element_size() * obj.nelement() / (1024 ** 2)
#                     total_mem += mem_mb
#                     print(f"Type: {type(obj).__name__:<20} "
#                           f"Shape: {tuple(obj.shape)} "
#                           f"Dtype: {obj.dtype} "
#                           f"Size: {mem_mb:.2f} MB")
#         except Exception:
#             pass  # Ignore inaccessible objects

#     print(f"\n📊 Total GPU Memory Used by Python Objects: {total_mem:.2f} MB")
#     print(f"📦 Total Allocated (PyTorch): {torch.cuda.memory_allocated() / (1024 ** 2):.2f} MB")
#     print(f"📦 Total Reserved (PyTorch): {torch.cuda.memory_reserved() / (1024 ** 2):.2f} MB\n")

# # Example usage:
# if __name__ == "__main__":
#     analyze_vram()


In [ ]:
# print(len(attr))
# print(len(attr[0]))
# print(len(attr[0][0]))
# print(len(attr[0][0][0]))
# print(len(delta))
# print(len(delta[0]))
# print(type(attr))
# print(type(attr[target_idx][sample_idx].tolist()))
# print(type(delta[target_idx][sample_idx].item()))
# print(attr[target_idx][sample_idx].tolist()[0])
# print(len(val_y))
# print("Prediction range:",
#       pred.min().item(),
#       pred.max().item())

# print("Target range:",
#       val_y.min().item(),
#       val_y.max().item())

# print("Mean |delta|:",
#       torch.stack(delta).abs().mean().item())

# print("Max |delta|:",
#       torch.stack(delta).abs().max().item())
# print(pred.shape)
# print(val_y.shape)

# print(torch.stack(delta).abs().mean())
# print(torch.stack(delta).abs().max())